In [6]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoProcessor, AutoModelForMultimodalLM
import os, sys, gc, torch

# hf
from dotenv import load_dotenv
import os
import boto3

load_dotenv(".env.test", override=True)

import torch, gc

# AWS
# s3 = boto3.client("s3")
# s3 = boto3.client("s3", region_name=os.getenv("AWS_DEFAULT_REGION"))
# bucket = os.getenv("BUCKET")
# prefix = os.getenv('MODELS_PREFIX')

cache_dir = os.getenv('CACHE_DIR')
# Set HF_TOKEN in your environment (RunPod secrets / terminal export), not in the notebook
token = os.environ.get("HF_TOKEN")
if not token:
    raise ValueError("Set HF_TOKEN before running (export HF_TOKEN=hf_...)")
from huggingface_hub import login

login(token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# load_dotenv(".env.test")
# print(os.environ.get('CACHE_DIR'))

True

In [8]:
gc.collect()
torch.cuda.empty_cache()

In [9]:
!nvidia-smi

Mon Jul 13 16:39:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:03:00.0  On |                  N/A |
|  0%   53C    P8             25W /  260W |    2427MiB /  11264MiB |     18%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:


processor = AutoProcessor.from_pretrained("unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit")
model = AutoModelForMultimodalLM.from_pretrained("unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))